In [1]:
#!pip install azure-functions-durable==1.2.9

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import logging
import sys
import os
sys.path.append('../')
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import azure.functions as func
import azure.durable_functions as df

In [6]:
bp = df.Blueprint()

## Internal working




In [7]:
myApp = df.DFApp(http_auth_level=func.AuthLevel.ANONYMOUS)

In [8]:
import azure.functions as func

We want to able to schedule these pipeline, so we wil create an instance of `runner` that will contains all those pipelines

> note: Is a table a proper a better way to storage this information?


The framework will have 2 `standard` functions:
1. a function that will have a timer trigger (60 minutes), that will check with the runner which pipelines needs to be executed

> note: we assume that this function will start each pipeline and with the control of [Configuration throttles](https://learn.microsoft.com/en-us/azure/azure-functions/durable/durable-functions-perf-and-scale#configuration-of-throttles) we will able to limit the concurrency, by default the framework enable one `pipeline` and one `task` each time. The important question is: Can we start all the pipelines needed and this configuration will manage the concurrency without any intervention?

also it seems nthat we need to take into account the `partitionCount`

> Note: If the user want to trigger a specific pipeline we will recommend to start trough rest API the same method `run_pipeline`. Also **Right now we asume that when the user perform that operation, the durable function framework add to the existing queue without any intervention.

2. A function to get the status of a pipeline -> this can be useful to leverage other external monitoring acitivities. The idea is to have a http trigger.

> can we also use this to get the status of the pipeline running?

In [9]:
import os
import datetime
from centraal_dataframework.tasks import TaskInfo, Task
from centraal_dataframework.pipeline import Pipeline
from centraal_dataframework.configuration import FrameworkConfiguration
from centraal_dataframework.runner import Runner

DUMMY_STRING = "DefaultEndpointsProtocol=https;AccountName=na;AccountKey=key!=k;EndpointSuffix=core.windows.net"
OVERWRITE_TAREA = os.environ.get("SOBREESCRIBIR_TAREA", True)
TAREA_DEBE_ESTAR_CONF = os.environ.get("TAREA_DEBE_CONFIGURACION", True)
UTC_OFFSET = datetime.timedelta(hours=os.environ.get("UTC_OFFSET", -5))


In [10]:
bp = df.Blueprint()

@Task
def first_step_of_pipeline(task_info: TaskInfo):
    print("first step")


@Task
def second_step_of_pipeline(task_info: TaskInfo):
    print("second step")


@Task
def other_step_of_pipeline(task_info: TaskInfo):
    print("second step")

pipeline = Pipeline(name="test_pipeline", tasks=[first_step_of_pipeline, second_step_of_pipeline], bp=bp)

otherpipeline = Pipeline(name="other_pipeline", tasks=[first_step_of_pipeline, other_step_of_pipeline, second_step_of_pipeline], bp=bp)

pipeline_test = Pipeline(name="test_pipeline_2", tasks=[first_step_of_pipeline], bp=bp)

In [11]:
app = func.FunctionApp(http_auth_level=func.AuthLevel.FUNCTION)


config = FrameworkConfiguration()
runner = Runner(pipelines = [pipeline, pipeline_test], conf=config)

app.register_functions(bp)
runner.register_framework_functions(app)


INFO:root:cargando configuracion desde centraal_dataframework.yaml


In [12]:
fs = app.get_functions()

for f in fs:
    print(f)

{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "name": "task_info", "activity": "alert"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "name": "task_info", "activity": "first_step_of_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "name": "task_info", "activity": "second_step_of_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "orchestrationTrigger", "name": "context", "orchestration": "test_pipeline"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "durableClient", "name": "client_test_pipeline"}, {"direction": "IN", "type": "httpTrigger", "name": "req", "authLevel": "FUNCTION", "route": "start_test_pipeline"}, {"direction": "OUT", "type": "http", "name": "$return"}]}
{"scriptFile": "function_app.py", "bindings": [{"direction": "IN", "type": "activityTrigger", "nam

In [13]:
timer = fs[-2].get_user_function()

In [14]:
from unittest.mock import MagicMock
import azure.functions._abc as func_abc
mock_timer_request = MagicMock(spec=func_abc.TimerRequest)
mock_timer_request.schedule = "*/5 * * * * *"
mock_timer_request.name = "test_timer"
mock_timer_request.past_due = False

timer(mock_timer_request)

ConnectionError: HTTPSConnectionPool(host='none.azurewebsites.net', port=443): Max retries exceeded with url: /api/start_test_pipeline (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7fac1b779490>: Failed to resolve 'none.azurewebsites.net' ([Errno -2] Name or service not known)"))